<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

## Step 7.0 — Setup

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

# Folder + file constants. _FOLDER = a directory, _FILE = a single file.
TUTORIALS_FOLDER = Path("tutorials")
DATA_FOLDER = TUTORIALS_FOLDER / "data"
SNAPSHOT_FOLDER = DATA_FOLDER / "snapshots"
CROP_FOLDER = DATA_FOLDER / "crops"
DB_FILE = DATA_FOLDER / "birds.db"
SAMPLE_BIRD_FILE = DATA_FOLDER / "samples" / "sample-bird.jpg"
MODEL_FILE = TUTORIALS_FOLDER / "yolov8n.pt"

# From .env (or hardcoded fallback for Colab)
PHONE_IP = os.environ.get("PHONE_IP", "192.168.1.42")
PHONE_URL = f"http://{PHONE_IP}:8080/photo.jpg"
SLACK_WEBHOOK = os.environ.get("SLACK_WEBHOOK", "")
HUGGINGFACE_API_KEY = os.environ.get("HUGGINGFACE_API_KEY", "")

SNAPSHOT_FOLDER.mkdir(parents=True, exist_ok=True)
CROP_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Snapshot folder: {SNAPSHOT_FOLDER}")
print(f"Phone URL: {PHONE_URL}")

from bird_watcher.save_sighting import save_sighting_to_db
print(f"Slack webhook set: {bool(SLACK_WEBHOOK)}")

Snapshot folder: tutorials/data/snapshots
Phone URL: http://192.168.1.207:8080/photo.jpg
Slack webhook set: False


## Step 7.1 — Slack messages use Block Kit

Slack Block Kit is JSON that describes the message layout. A header + a section with fields is enough for a sighting alert.

In [ ]:
payload = {
    "blocks": [
        {
            "type": "header",
            "text": {"type": "plain_text", "text": ":bird: New sighting — Northern Cardinal"},
        },
        {
            "type": "section",
            "fields": [
                {"type": "mrkdwn", "text": "*Species*\nNorthern Cardinal"},
                {"type": "mrkdwn", "text": "*Confidence*\n92%"},
            ],
        },
    ]
}
import json
print(json.dumps(payload, indent=2))

{
  "blocks": [
    {
      "type": "header",
      "text": {
        "type": "plain_text",
        "text": ":bird: New sighting \u2014 Northern Cardinal"
      }
    },
    {
      "type": "section",
      "fields": [
        {
          "type": "mrkdwn",
          "text": "*Species*\nNorthern Cardinal"
        },
        {
          "type": "mrkdwn",
          "text": "*Confidence*\n92%"
        }
      ]
    }
  ]
}


## Step 7.2 — Wrap the payload builder as `build_sighting_alert`

In [0]:
#| echo: false
#| output: asis
show_doc(build_sighting_alert)

---

### build_sighting_alert

```python
def build_sighting_alert(
    species:str, confidence:float, snapshot_file:Path, sighting_id:int | None=None
)->dict:
```

*Build a Slack Block Kit message payload for one sighting.*

Args:
    species: the bird's name.
    confidence: 0.0 - 1.0.
    snapshot_file: the photo to attach.
    sighting_id: optional database id for cross-referencing.

Returns:
    A dict ready to be POSTed as JSON to a Slack webhook.

## Step 7.3 — Add `send_alert_to_slack`

In [0]:
#| echo: false
#| output: asis
show_doc(send_alert_to_slack)

---

### send_alert_to_slack

```python
def send_alert_to_slack(
    payload:dict, webhook_url:str | None=None, verbose:bool=True
)->bool:
```

*POST a message payload to Slack. Prints locally if no webhook.*

Args:
    payload: the dict returned by `build_sighting_alert`.
    webhook_url: the Slack incoming webhook URL. If None, falls back
        to the SLACK_WEBHOOK env var.
    verbose: if True, print whether we sent or just previewed.

Returns:
    True if the message was sent (or dry-run previewed), False if failed.

## Acceptance criterion

In [ ]:
from bird_watcher.send_alert import build_sighting_alert, send_alert_to_slack

jpg_files = sorted(SNAPSHOT_FOLDER.glob("*.jpg"))
snapshot_file = SNAPSHOT_FOLDER / jpg_files[0].name if jpg_files else SNAPSHOT_FOLDER / "missing.jpg"

row_id = save_sighting_to_db(DB_FILE, snapshot_file, "Northern Cardinal", 0.92, verbose=False)
payload = build_sighting_alert("Northern Cardinal", 0.92, snapshot_file, sighting_id=row_id)
sent = send_alert_to_slack(payload)
assert sent, "send_alert_to_slack should return True (dry-run if no webhook)"
print("✅ Notification flow works")

[dry-run] No SLACK_WEBHOOK set. Would have sent:
{
  "blocks": [
    {
      "type": "header",
      "text": {
        "type": "plain_text",
        "text": ":bird: New sighting \u2014 Northern Cardinal (#3)"
      }
    },
    {
      "type": "section",
      "fields": [
        {
          "type": "mrkdwn",
          "text": "*Species*\nNorthern Cardinal"
        },
        {
          "type": "mrkdwn",
          "text": "*Confidence*\n92%"
        },
        {
          "type": "mrkdwn",
          "text": "*Photo*\n`2026-07-07_23-46-57.jpg`"
        }
      ]
    }
  ]
}
✅ Notification flow works


## What's next

**Step 8:** open [08-web-hello.ipynb](08-web-hello.ipynb) — we'll start a tiny web app with FastAPI.